# Session 9: Bootcamp: Comparing Classifiers for a Bank Marketing Campaign

## Overview

A Portuguese bank sells term deposits by phoning its customers. Most calls fail: only about 1 in 9 customers
says yes. If the bank could **rank customers by their chance of subscribing**, it could call the most promising
ones first and spend far less time on calls that go nowhere.

This bootcamp puts Part 2 together on a realistic table of mixed data. You will find and remove a leak, handle
missing values and categories in a single preprocessing pipeline, and compare **seven classifiers** fairly: a
dummy baseline, logistic regression, k-nearest neighbours, a decision tree, naive Bayes, a random forest and
gradient boosting. Then you tune the best one and express the result in the terms the marketing team cares
about: how many subscribers they reach for a given number of calls.

By the end, you will be able to:

- Recognise a leak that is only known *after* the outcome.
- Build one `ColumnTransformer` that imputes, encodes and scales mixed columns.
- Compare many classifiers with the same folds and the same metrics, and explain their strengths.
- Explain why accuracy is useless on an imbalanced problem, and use ROC AUC and average precision instead.
- Tune a model with a small grid search, and report the result as a gains table.

**Prerequisites:** Sessions 3 (tree ensembles), 5 (logistic regression, precision and recall), 7 (pipelines,
grid search, average precision).

**Dataset:** [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing). 45,211 phone contacts
from a Portuguese bank's direct-marketing campaigns, May 2008 to November 2010 (Moro et al., 2014): the customer's
age, job, marital status, education, credit default, average balance, housing and personal loans; details of the
contact; the customer's history with earlier campaigns; and whether they subscribed. It is downloaded automatically
by the code.

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers **← you are here** |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** Which classifier ranks customers best, and how many subscribers can the bank reach
with a fraction of the calls?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| ucimlrepo | Downloads the dataset from UCI |
| scikit-learn | Preprocessing, seven classifiers, cross-validation, grid search, metrics |
| pandas | Tables |
| numpy | Arrays |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, accuracy_score, average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from importlib.metadata import version

for package in ["ucimlrepo", "scikit-learn", "pandas", "numpy", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the data and read the column descriptions

In [ ]:
from ucimlrepo import fetch_ucirepo

bank = fetch_ucirepo(id=222)
print(bank.variables[["name", "type", "description"]].to_string(max_colwidth=90))

calls = bank.data.original
print(f"\n{len(calls):,} contacts; subscribed: {(calls['y'] == 'yes').mean():.1%}")

**Read the output:** 16 inputs of mixed types: numbers (age, balance, campaign...), categories (job, education,
contact...) and yes/no flags. Only **11.7%** of contacts ended in a subscription: the classes are imbalanced.

Read the description of `duration` carefully. Step 3 is about it.

## Step 3: The leak: call duration

`duration` is the length of the call in seconds. The UCI description itself warns that it "highly affects the
output target" but "is not known before a call is performed". Look at the subscription rate by call length.

In [ ]:
subscribed = (calls["y"] == "yes").astype(int)
bands = pd.cut(calls["duration"], [0, 60, 120, 300, 600, 5000], include_lowest=True)
print(subscribed.groupby(bands, observed=True).agg(["mean", "count"]).round(3)
      .rename(columns={"mean": "subscription rate"}))

**Read the output:** almost nobody who hangs up within a minute subscribes (0.2%), while nearly half of the calls
longer than 10 minutes end in a subscription (48%). Duration is a superb predictor, and completely useless: the
bank wants to choose *whom to call*, and a call's length is only known after it has been made. (Also, a long call
is partly a *consequence* of the customer being interested.)

We drop it. Step 10 measures how good the model would *look* with it.

## Step 4: Data quirks

Every real dataset has a few. Check missing values and the less obvious columns.

In [ ]:
print("missing values:")
print(calls.isna().sum()[lambda s: s > 0].to_string())
print(f"\npdays == -1 (never contacted before): {(calls['pdays'] == -1).mean():.1%}")
print(f"'day_of_week' values: {calls['day_of_week'].min()} to {calls['day_of_week'].max()}")

**Read the output:**

- **Missing values** in four categorical columns. `poutcome` (the outcome of the previous campaign) is missing for
  most customers, because most were never contacted before; `contact` is missing for 13,020. For a category,
  "missing" is information in itself, so we fill it with the value `"unknown"` and let it be a category of its own.
- **`pdays`** (days since the previous campaign's contact) is -1 for 81.7% of customers: a code meaning "never
  contacted", not a number of days. Tree models handle such codes well; for linear models it would be cleaner to
  add a flag. We keep it as it is here.
- **`day_of_week`** runs from 1 to 31: despite its name, it is the **day of the month**. Reading the values, not
  just the names, catches mistakes like this.

## Step 5: Split, and build one preprocessing step for every model

The pipeline's `ColumnTransformer` treats the two kinds of column differently:

- **categorical:** fill missing values with `"unknown"`, then one-hot encode;
- **numeric:** standardise (logistic regression, k-nearest neighbours and naive Bayes depend on scale; trees do
  not, but it does them no harm).

The split is stratified, so training and test sets both have 11.7% subscribers.

In [ ]:
X = calls.drop(columns=["y", "duration"])
y = subscribed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

CATEGORICAL = X.select_dtypes("object").columns.tolist()
NUMERIC = X.select_dtypes("number").columns.tolist()
print("categorical:", CATEGORICAL)
print("numeric:    ", NUMERIC)


def preprocess():
    return ColumnTransformer([
        ("categorical", make_pipeline(SimpleImputer(strategy="constant", fill_value="unknown"),
                                      OneHotEncoder(handle_unknown="ignore", sparse_output=False)), CATEGORICAL),
        ("numeric", StandardScaler(), NUMERIC),
    ])


print(f"\n{len(X_train):,} training rows -> {preprocess().fit_transform(X_train).shape[1]} model inputs")

**Read the output:** 9 categorical and 6 numeric columns become 50 model inputs after one-hot encoding.
`handle_unknown="ignore"` means a category seen for the first time in new data is encoded as all zeros rather than
causing an error.

## Step 6: Seven classifiers, compared fairly

Every model gets the **same preprocessing**, the **same five folds** and the **same metrics**:

- **ROC AUC:** the probability that a random subscriber is ranked above a random non-subscriber. 0.5 is guessing,
  1.0 is perfect.
- **Average precision (AP):** Session 7's ranking metric, focused on the top of the list. Guessing scores the base
  rate, 0.117.

The contenders:

| Model | How it decides |
| --- | --- |
| Dummy | Ignores the inputs; the baseline |
| Logistic regression | A weighted sum of inputs through a sigmoid (Session 5) |
| k-nearest neighbours | Share of subscribers among the k most similar training customers |
| Decision tree | A sequence of yes/no questions (Session 3) |
| Gaussian naive Bayes | Bayes' rule, assuming each input is normally distributed and independent within a class (Session 6) |
| Random forest | Average of many deep trees on bootstrap samples (Session 3) |
| Gradient boosting | Small trees added one after another, each fixing the last ones' errors (Session 3) |

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
classifiers = {
    "dummy": DummyClassifier(),
    "logistic regression": LogisticRegression(max_iter=2000),
    "k-nearest neighbours (k=50)": KNeighborsClassifier(n_neighbors=50),
    "decision tree (depth 6)": DecisionTreeClassifier(max_depth=6, random_state=0),
    "Gaussian naive Bayes": GaussianNB(),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
rows = {}
for name, classifier in classifiers.items():
    start = time.time()
    scores = cross_validate(make_pipeline(preprocess(), classifier), X_train, y_train, cv=cv,
                            scoring=["roc_auc", "average_precision"])
    rows[name] = {"ROC AUC": scores["test_roc_auc"].mean(), "AUC spread": scores["test_roc_auc"].std(),
                  "AP": scores["test_average_precision"].mean(), "seconds": time.time() - start}
comparison = pd.DataFrame(rows).T.sort_values("ROC AUC", ascending=False)
print(comparison.round(3))

**Read the output:** this takes 20 seconds or so.

- **Gradient boosting and the random forest are joint best** (ROC AUC 0.796 and 0.795, AP 0.441 and 0.442): almost four times
  the base rate at the top of the ranking. They differ by less than the fold-to-fold spread.
- **Logistic regression and k-nearest neighbours** are next (AUC 0.763 and 0.761), a respectable result for much
  simpler models.
- **Naive Bayes** (0.740) suffers from its assumptions: one-hot columns are not normally distributed, and the
  inputs are not independent.
- A **single decision tree** is the weakest real model (0.702), as Session 3 predicted.
- The **dummy** scores exactly 0.5 AUC and AP equal to the base rate, as it should.

The two ensembles are tied; gradient boosting trains a little faster, and is easier to tune. We continue with it.

## Step 7: Why not accuracy?

What would accuracy have said?

In [ ]:
for name in ["dummy", "logistic regression", "gradient boosting"]:
    model = make_pipeline(preprocess(), classifiers[name]).fit(X_train, y_train)
    print(f"{name:20s} accuracy = {accuracy_score(y_test, model.predict(X_test)):.3f}")

**Read the output:** the dummy, which predicts "no" for everyone, scores **88.3%** accuracy. The real models
barely beat it (89.3% and 89.6%). Accuracy at the default threshold of 0.5 rewards saying "no", and says nothing
about how well a model ranks. On imbalanced problems, compare models with ROC AUC or AP.

## Step 8: Tune gradient boosting

A small grid over the settings that matter most for gradient boosting: the **learning rate** and the size of
each tree (**max_leaf_nodes**). We score with AP, since the bank cares about the top of the list.

In [ ]:
pipeline = make_pipeline(preprocess(), HistGradientBoostingClassifier(max_iter=300, random_state=0))
grid = {"histgradientboostingclassifier__learning_rate": [0.03, 0.1],
        "histgradientboostingclassifier__max_leaf_nodes": [15, 31, 63]}
search = GridSearchCV(pipeline, grid, cv=cv, scoring="average_precision", n_jobs=-1).fit(X_train, y_train)

results = pd.DataFrame(search.cv_results_["params"]).rename(columns=lambda c: c.split("__")[1])
results["CV AP"] = search.cv_results_["mean_test_score"]
print(results.sort_values("CV AP", ascending=False).round(4).to_string(index=False))

**Read the output:** the six settings are within 0.005 AP of each other, and the best (learning rate 0.1, 15
leaves) scores 0.444, hardly better than the defaults in Step 6 (0.441). That is typical: gradient boosting's defaults are good, and the choice of
inputs and the handling of leaks matter far more than tuning. `search.best_estimator_` is the tuned pipeline,
refitted on all the training data.

## Step 9: The result in the bank's terms

Score the test customers once. Then turn the ranking into a **gains table**: if the bank calls only the top 10%,
20%, ... of customers by predicted probability, what share of all the subscribers does it reach?

In [ ]:
best = search.best_estimator_
p_test = best.predict_proba(X_test)[:, 1]
print(f"test ROC AUC = {roc_auc_score(y_test, p_test):.3f}   test AP = {average_precision_score(y_test, p_test):.3f}")

order = np.argsort(-p_test)
reached = np.cumsum(y_test.to_numpy()[order]) / y_test.sum()
gains = pd.DataFrame({"share of customers called": [0.1, 0.2, 0.3, 0.5],
                      "share of subscribers reached": [reached[int(f * len(order)) - 1] for f in [0.1, 0.2, 0.3, 0.5]]})
gains["lift"] = gains["share of subscribers reached"] / gains["share of customers called"]
print(gains.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
RocCurveDisplay.from_predictions(y_test, p_test, ax=axes[0], name="tuned gradient boosting")
axes[0].plot([0, 1], [0, 1], "k--", label="guessing")
axes[0].legend()
axes[1].plot(np.arange(1, len(order) + 1) / len(order), reached, label="model ranking")
axes[1].plot([0, 1], [0, 1], "k--", label="random calling")
axes[1].set_xlabel("share of customers called")
axes[1].set_ylabel("share of subscribers reached")
axes[1].set_title("Gains chart")
axes[1].legend()
plt.show()

**Read the output:** the test AUC (0.802) and AP (0.462) agree with cross-validation. The gains table is the
result to show the marketing team: calling the **top 10%** of customers reaches 44% of all subscribers; the **top
20%** reaches 62%, and the top 30% reaches 72%. Random calling would reach 10%, 20% and 30%. **Lift** is the ratio: how many times more
subscribers per call than random calling.

Where to stop calling is a business decision: it depends on the cost of a call and the value of a deposit. The
model's job is to put the list in the best order.

## Step 10: What drives the ranking, and the price of the leak

Permutation importance on the test set (scored by ROC AUC), and, for comparison, the same model trained *with*
the leaky `duration` column.

In [ ]:
perm = permutation_importance(best, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=0, n_jobs=-1)
print(pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).round(3)
      .head(8).to_frame("AUC drop when scrambled").to_string())

X_leaky = calls.drop(columns="y")
leaky = make_pipeline(
    ColumnTransformer([("categorical", make_pipeline(SimpleImputer(strategy="constant", fill_value="unknown"),
                                                     OneHotEncoder(handle_unknown="ignore")), CATEGORICAL)],
                      remainder="passthrough"),
    HistGradientBoostingClassifier(random_state=0))
leaky_scores = cross_validate(leaky, X_leaky.loc[X_train.index], y_train, cv=cv, scoring="roc_auc")
print(f"\nCV ROC AUC with duration:    {leaky_scores['test_score'].mean():.3f}")
print(f"CV ROC AUC without duration: {comparison.loc['gradient boosting', 'ROC AUC']:.3f}")

**Read the output:** the most important inputs describe **how and when** the customer was contacted, and their
history, not their profile: `contact` (mobile, landline or unknown; scrambling it costs 0.09 AUC), `month`, the day
of the month, and `poutcome` (whether the customer subscribed in a previous campaign). Campaign timing and method
varied a lot over the two and a half years, and customers who said yes before tend to say yes again. Housing
loans, balance and age matter less.

With `duration`, the model "achieves" a ROC AUC of 0.935 instead of 0.796. Anyone shown that number would
expect a near-perfect call list, and would get one no better than our honest model, since duration is not
available when the list is drawn up.

## Bootcamp summary

| Step | Decision | Why |
| --- | --- | --- |
| Columns | Drop `duration` | Only known after the call: leakage (AUC 0.93 vs 0.80) |
| Missing values | `"unknown"` category | Missing is informative for categories |
| Metric | ROC AUC and AP, not accuracy | The dummy is 88% accurate |
| Comparison | Seven models, same folds | Ensembles best; logistic regression a strong simple alternative |
| Model | Tuned gradient boosting | Joint best, fastest of the two ensembles |
| Result | Top 20% of calls reach 62% of the subscribers | Gains table for the business |

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Keeping inputs measured during or after the outcome | Excellent CV scores, useless in practice | Ask of every column: is it known at prediction time? |
| Dropping rows with missing categories | Loses data and hides a useful signal | Impute `"unknown"` inside the pipeline |
| Trusting column names | `day_of_week` is really the day of the month | Look at the values |
| Judging an imbalanced problem by accuracy | A model that says "no" to everyone looks good | ROC AUC, AP, gains tables |
| Different folds or preprocessing for different models | Unfair comparison | One `cv` object and one `preprocess()` for all |
| Spending hours on tuning | Small gains | Fix the data and inputs first; tune a little at the end |

## Exercises

- [ ] The rows are in date order. Re-run the comparison using the last 20% of rows as the test set, with no
  shuffling. What happens to the scores, and why might this be the more honest test?
- [ ] Add `class_weight="balanced"` to logistic regression. Does its ROC AUC change? Does its accuracy? Explain.
- [ ] Add a linear SVM (`LinearSVC`) to the comparison, using `scoring` on `decision_function` (cross_validate
  handles this automatically for AUC).
- [ ] Find the share of customers to call so that the expected profit is highest, if a call costs €5 and a
  subscription is worth €60.

This ends Part 2. Next, Session 10 starts Part 3: clustering, where there are no labels at all, beginning with
k-means built from scratch.